# Fase 2d: mitigación de falsos positivos, falsos negativos y confusión con roya
Tesis: aplicación móvil para identificar síntomas de ojo de gallo (Rodrigo Alfaro, UCC).

**Qué se prueba.** Tres medidas para reducir errores, evaluadas sin usar para entrenar las hojas con las que se mide:
1. **Diagnóstico v3 "ojo de gallo frente a todo lo demás"**: la roya entra como ejemplo de "no es ojo de gallo", para
   reducir los falsos positivos por roya, conservando una sola salida (compatible con la aplicación).
2. **Diversidad de origen**: además de Perú se entrena con hojas de PG 26038 (Costa Rica). Para medir sin trampa se usa
   **validación cruzada por hoja en 5 rondas**: en cada ronda se reserva 1/5 de las hojas de PG 26038 (las dos caras de
   una hoja siempre van juntas) y se evalúa solo con ellas.
3. **Promedio de cuatro vistas (TTA)**: la imagen original, volteada horizontal, volteada vertical y girada 180°.

Se compara con el diagnóstico actual (v1) en las mismas hojas. El modelo final se entrena con todos los datos.
**Cómo usarlo:** Entorno de ejecución → Cambiar tipo de entorno → T4 GPU. Luego Ejecutar todas (unos 30 minutos).

In [1]:
import os, sys, json, math, time, random, platform, subprocess
MOCK = os.environ.get("MOCK") == "1"
if not MOCK:
    from google.colab import drive
    drive.mount("/content/drive")
import numpy as np, pandas as pd, tensorflow as tf, keras
from PIL import Image, ImageOps
from concurrent.futures import ThreadPoolExecutor
from sklearn.metrics import roc_auc_score

SEMILLA = 42; TAM = 224
EPOCAS = int(os.environ.get("EPOCAS", 15)); K = 5
BASE = os.environ.get("RUTA_BASE", "/content/drive/MyDrive/tesis_ojo_de_gallo")
DS = os.path.join(BASE, "02_Dataset")
TMP = "/content/tmp_fase2d" if not MOCK else "/tmp/tmp_fase2d"
RUTAS = {"ojo_de_gallo": os.path.join(DS, "Ojo_Gallo"), "sana": os.path.join(DS, "Sanas"), "roya": os.path.join(DS, "Roya"),
         "pg26038": os.path.join(DS, "externos", "PG26038_muestras", "images"),
         "diag_v1": os.path.join(BASE, "fase2_resultados", "android", "modelo_diagnostico.tflite")}
SAL = os.path.join(BASE, "fase2d_mitigacion")
for sub in ["tablas", "android"]:
    os.makedirs(os.path.join(SAL, sub), exist_ok=True)
os.makedirs(TMP, exist_ok=True)
EXT = (".jpg", ".jpeg", ".png")
def fijar_semillas(s=SEMILLA):
    random.seed(s); np.random.seed(s); keras.utils.set_random_seed(s)
fijar_semillas()
VERSIONES = {"python": platform.python_version(), "tensorflow": tf.__version__, "keras": keras.__version__,
             "gpu": [d.name for d in tf.config.list_physical_devices("GPU")]}
print(VERSIONES)
for k, r in RUTAS.items():
    print(("OK   " if os.path.exists(r) else "FALTA"), k, r)
def guardar_json(obj, nombre):
    with open(os.path.join(SAL, "tablas", nombre), "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=float)
def wilson(k, n, z=1.959963984540054):
    if n == 0: return (float("nan"), float("nan"))
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))

Mounted at /content/drive
{'python': '3.13.16', 'tensorflow': '2.21.0', 'keras': '3.13.2', 'gpu': ['/physical_device:GPU:0']}
OK    ojo_de_gallo /content/drive/MyDrive/tesis_ojo_de_gallo/02_Dataset/Ojo_Gallo
OK    sana /content/drive/MyDrive/tesis_ojo_de_gallo/02_Dataset/Sanas
OK    roya /content/drive/MyDrive/tesis_ojo_de_gallo/02_Dataset/Roya
OK    pg26038 /content/drive/MyDrive/tesis_ojo_de_gallo/02_Dataset/externos/PG26038_muestras/images
OK    diag_v1 /content/drive/MyDrive/tesis_ojo_de_gallo/fase2_resultados/android/modelo_diagnostico.tflite


## 1. Lectura (igual que la aplicación) y conjuntos

In [2]:
def leer(ruta):
    with Image.open(ruta) as im:
        im = ImageOps.exif_transpose(im)
        if im.mode in ("RGBA", "LA") or (im.mode == "P" and "transparency" in im.info):
            im = Image.alpha_composite(Image.new("RGBA", im.size, (255, 255, 255, 255)), im.convert("RGBA"))
        arr = np.asarray(im.convert("RGB"), dtype=np.float32)
    return np.clip(np.round(tf.image.resize(arr[None], (TAM, TAM), method="bilinear").numpy()[0]), 0, 255).astype("uint8")
def leer_varias(rutas, desc):
    t0 = time.time()
    def seguro(r):
        try: return leer(r)
        except Exception as e: print("dañada:", os.path.basename(r), e); return None
    with ThreadPoolExecutor(8) as ex: res = list(ex.map(seguro, rutas))
    ok = [r for r, x in zip(rutas, res) if x is not None]
    print(f"{desc}: {len(ok)} de {len(rutas)} en {time.time() - t0:.0f} s")
    return np.stack([x for x in res if x is not None]), ok
def listar(c):
    return sorted(os.path.join(c, f) for f in os.listdir(c) if f.lower().endswith(EXT)) if os.path.isdir(c) else []
def ds_desde(X, y, entrenar):
    d = tf.data.Dataset.from_tensor_slices((X, y))
    if entrenar: d = d.shuffle(len(X), seed=SEMILLA, reshuffle_each_iteration=True)
    return d.batch(32).map(lambda a, b: (tf.cast(a, tf.float32), b), num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

PARTICION = json.loads(r'''{"train": {"ojo_de_gallo": ["20251129_091959.jpg", "20251129_092016.jpg", "20251129_092053.jpg", "20251129_092106.jpg", "20251129_092123.jpg", "20251129_092139.jpg", "20251129_092149.jpg", "20251129_092158.jpg", "20251129_092220.jpg", "20251129_092252.jpg", "20251129_092438.jpg", "20251129_092457.jpg", "20251129_092504.jpg", "20251129_092511.jpg", "20251129_092539.jpg", "20251129_092559.jpg", "20251129_092702.jpg", "20251129_092708.jpg", "20251129_092800.jpg", "20251129_092931.jpg", "20251129_093055.jpg", "20251129_093132.jpg", "20251129_093207.jpg", "20251129_093218.jpg", "20251129_093257.jpg", "20251129_093304.jpg", "20251129_093324.jpg", "20251129_093359.jpg", "20251129_093502.jpg", "20251129_093615.jpg", "20251129_093623.jpg", "20251129_093725.jpg", "20251129_093729.jpg", "20251129_093736.jpg", "20251129_093956.jpg", "20251129_094006.jpg", "20251129_094032.jpg", "20251129_094102.jpg", "20251129_094107.jpg", "20251129_094130.jpg", "20251129_094219.jpg", "20251129_094307.jpg", "20251129_094337.jpg", "20251129_094614.jpg", "20251129_094622.jpg", "20251129_094627.jpg", "20251129_094654.jpg", "20251129_094701.jpg", "20251129_094834.jpg", "20251129_094950.jpg", "20251129_095011.jpg", "20251129_095106.jpg", "20251129_095132.jpg", "20251129_095138.jpg", "20251129_095206.jpg", "20251129_095244.jpg", "20251129_095331.jpg", "20251129_095408.jpg", "20251129_095416.jpg", "20251129_095449.jpg", "20251129_095512.jpg", "20251129_095538.jpg", "20251129_095552.jpg", "20251129_095556.jpg", "20251129_095601.jpg", "20251129_095609.jpg", "20251129_095629.jpg", "20251129_095642.jpg", "20251129_095712.jpg", "20251129_095729.jpg", "20251129_095736.jpg", "20251129_095755.jpg", "20251129_095810.jpg", "20251129_095821.jpg", "20251129_100046.jpg", "20251129_100416.jpg", "20251129_100436.jpg", "20251129_100506.jpg", "20251129_100515.jpg", "20251129_100535.jpg", "20251129_100547.jpg", "20251129_100621.jpg", "20251129_100711.jpg", "20251129_100808.jpg", "20251129_100906.jpg", "20251129_100918.jpg", "20251129_100948.jpg", "20251129_100955.jpg", "20251129_101023.jpg", "20251129_101029.jpg", "20251129_101033.jpg", "20251129_101157.jpg", "20251129_101255.jpg", "20251129_101307.jpg", "20251129_101319.jpg", "20251129_101337.jpg", "20251129_101342.jpg", "20251129_101430.jpg", "20251129_101441.jpg", "20251129_101504.jpg", "20251129_101509.jpg", "20251129_101518.jpg", "20251129_101529.jpg", "20251129_101559.jpg", "20251129_101610.jpg", "20251129_101624.jpg", "20251129_101641.jpg", "20251129_101714.jpg", "20251129_101831.jpg", "20251129_101839.jpg", "20251129_101856.jpg", "20251129_101905.jpg", "20251129_102019.jpg", "20251129_102125.jpg", "20251129_102402.jpg", "20251129_102518.jpg", "20251129_102657.jpg", "20251129_102737.jpg", "20251129_102838.jpg", "20251129_102905.jpg", "20251129_103003.jpg", "20251129_103051.jpg", "20251129_103124.jpg", "20251129_103139.jpg", "20251129_103154.jpg", "20251129_103206.jpg", "20251129_103304.jpg", "20251129_103311.jpg", "20251129_103353.jpg", "20251129_103409.jpg", "20251129_103437.jpg", "20251129_103445.jpg", "20251129_103532.jpg", "20251129_103542.jpg", "20251129_103604.jpg", "20251129_103618.jpg", "20251129_103647.jpg", "20251129_103728.jpg", "20251129_103758.jpg", "20251129_103804.jpg", "20251129_103839.jpg", "20251129_103847.jpg", "20251129_103938.jpg", "20251129_104212.jpg", "20251129_104314.jpg", "20251129_104320.jpg", "20251129_104346.jpg", "20251129_104351.jpg", "20251129_104356.jpg", "20251129_104409.jpg", "20251129_104557.jpg", "20251129_104706.jpg", "20251129_104727.jpg", "20251129_104745.jpg", "20251129_104803.jpg", "20251129_104901.jpg", "20251129_104924.jpg", "20251129_104958.jpg", "20251129_105257.jpg", "20251129_105404.jpg", "20251129_105435.jpg", "20251129_105505.jpg", "20251129_105517.jpg", "20251129_105537.jpg", "20251129_114546.jpg", "20251129_114556.jpg", "20251129_114630.jpg", "20251129_114655.jpg", "20251129_114830.jpg", "20251129_114858.jpg", "20251129_114932.jpg", "20251129_114957.jpg", "20251129_115010.jpg", "20251129_115017.jpg", "20251129_115123.jpg", "20251129_115140.jpg", "20251129_115201.jpg", "20251129_115218.jpg", "20251129_115223.jpg", "20251129_115251.jpg", "20251129_115305.jpg", "20251129_115313.jpg", "20251129_115322.jpg", "20251129_115331.jpg", "20251129_115426.jpg", "20251129_115503.jpg", "20251129_115534.jpg", "20251129_115633.jpg", "20251129_115719.jpg", "20251129_115734.jpg", "20251129_115754.jpg", "20251129_115815.jpg", "20251129_115822.jpg", "20251129_115827.jpg", "20251129_115900.jpg", "20251129_115923.jpg", "20251129_120045.jpg", "20251129_120152.jpg", "20251129_120217.jpg", "20251129_120227.jpg", "20251129_120259.jpg", "20251129_120417.jpg", "20251129_120512.jpg", "20251129_120518.jpg", "20251129_120531.jpg", "20251129_120544.jpg", "20251129_120552.jpg", "20251129_120745.jpg", "20251129_120748.jpg", "20251129_120803.jpg", "20251129_120820.jpg", "20251129_120831.jpg", "20251129_120901.jpg", "20251129_120915.jpg", "20251129_120956.jpg", "20251129_121017.jpg", "20251129_121030.jpg", "20251129_121051.jpg", "20251129_121134.jpg", "20251129_130238.jpg", "20251129_130306.jpg", "20251129_130402.jpg", "20251129_130457.jpg", "20251129_130522.jpg", "20251129_130644.jpg", "20251129_130715.jpg", "20251129_130723.jpg", "20251129_130733.jpg", "20251129_130802.jpg", "20251129_130807.jpg", "20251129_130817.jpg", "20251129_130850.jpg", "20251129_131040.jpg", "20251129_131046.jpg", "20251129_131056.jpg", "20251129_131125.jpg", "20251129_131202.jpg", "20251129_131320.jpg", "20251129_131514.jpg", "20251129_131521.jpg", "20251129_131525.jpg", "20251129_131529.jpg", "20251129_131655.jpg", "20251129_131720.jpg", "20251129_131750.jpg", "20251129_131815.jpg", "20251129_131820.jpg", "20251129_131839.jpg", "20251129_131901.jpg", "20251129_131914.jpg", "20251129_131938.jpg", "20251129_132038.jpg", "20251129_132125.jpg", "20251129_132200.jpg", "20251129_132354.jpg", "20251129_132449.jpg", "20251129_132501.jpg", "20251129_132528.jpg", "20251129_132534.jpg", "20251129_132732.jpg", "20251129_132751.jpg", "20251129_132832.jpg", "20251129_132854.jpg", "20251129_132915.jpg", "20251129_132921.jpg", "20251129_132954.jpg", "20251129_133000.jpg", "20251129_133018.jpg", "20251129_133340.jpg", "20251129_133353.jpg", "20251129_133423.jpg", "20251129_133456.jpg", "20251129_133515.jpg", "20251129_133519.jpg", "20251129_133625.jpg", "20251129_133633.jpg", "20251129_133740.jpg", "20251129_133814.jpg", "20251129_133850.jpg", "20251129_133912.jpg", "20251129_133953.jpg", "20251129_133959.jpg", "20251129_134106.jpg", "20251129_134335.jpg", "20251129_134401.jpg", "20251129_134420.jpg", "20251129_134502.jpg", "20251129_134506.jpg", "20251129_134517.jpg", "20251129_134527.jpg", "20251129_134557.jpg", "20251129_134615.jpg", "20251129_134713.jpg", "20251129_134724.jpg", "20251129_135000.jpg", "20251129_135004.jpg", "20251129_135020.jpg", "20251129_135027.jpg", "20251129_135036.jpg", "20251129_135042.jpg", "20251129_135118.jpg", "20251129_135142.jpg", "20251129_135152.jpg", "20251129_135205.jpg", "20251129_135212.jpg", "20251129_135217.jpg", "20251129_135225.jpg", "20251129_135235.jpg", "20251129_135253.jpg", "20251129_135335.jpg", "20251129_135341.jpg", "20251129_135422.jpg", "20251129_135428.jpg", "20251129_135448.jpg", "20251129_135510.jpg", "20251129_135538.jpg", "20251129_135551.jpg", "20251129_135556.jpg", "20251129_135755.jpg", "20251129_135804.jpg", "20251129_135827.jpg", "20251129_135845.jpg", "20251129_140008.jpg", "20251129_140011.jpg", "20251129_140020.jpg", "20251129_140038.jpg", "20251129_140128.jpg", "20251129_140136.jpg", "20251129_140207.jpg", "20251129_140216.jpg", "20251129_140227.jpg", "20251129_140240.jpg", "20251129_140502.jpg", "20251129_140505.jpg", "20251129_140513.jpg", "20251129_140528.jpg", "20251129_140538.jpg", "20251129_140553.jpg", "20251129_140618.jpg", "20251129_140653.jpg", "20251129_140657.jpg", "20251129_140708.jpg", "20251129_140713.jpg", "20251129_140744.jpg", "20251129_140816.jpg", "20251129_140820.jpg", "20251129_140842.jpg", "20251129_140850.jpg", "20251129_140907.jpg", "20251129_140938.jpg"], "sana": ["Sana 1.jpg", "Sana 100.jpg", "Sana 103.jpg", "Sana 106.jpg", "Sana 107.jpg", "Sana 109.jpg", "Sana 110.jpg", "Sana 111.jpg", "Sana 114.jpg", "Sana 115.jpg", "Sana 118.jpg", "Sana 121.jpg", "Sana 124.jpg", "Sana 125.jpg", "Sana 127.jpg", "Sana 129.jpg", "Sana 131.jpg", "Sana 138.jpg", "Sana 14.jpg", "Sana 141.jpg", "Sana 143.jpg", "Sana 147.jpg", "Sana 148.jpg", "Sana 15.jpg", "Sana 154.jpg", "Sana 156.jpg", "Sana 163.jpg", "Sana 169.jpg", "Sana 173.jpg", "Sana 178.jpg", "Sana 18.jpg", "Sana 183.jpg", "Sana 188.jpg", "Sana 192.jpg", "Sana 194.jpg", "Sana 199.jpg", "Sana 20.jpg", "Sana 201.jpg", "Sana 203.jpg", "Sana 205.jpg", "Sana 206.jpg", "Sana 207.jpg", "Sana 208.jpg", "Sana 214.jpg", "Sana 221.jpg", "Sana 222.jpg", "Sana 223.jpg", "Sana 224.jpg", "Sana 225.jpg", "Sana 227.jpg", "Sana 237.jpg", "Sana 240.jpg", "Sana 241.jpg", "Sana 243.jpg", "Sana 244.jpg", "Sana 245.jpg", "Sana 249.jpg", "Sana 251.jpg", "Sana 253.jpg", "Sana 254.jpg", "Sana 255.jpg", "Sana 258.jpg", "Sana 259.jpg", "Sana 260.jpg", "Sana 261.jpg", "Sana 264.jpg", "Sana 267.jpg", "Sana 268.jpg", "Sana 269.jpg", "Sana 27.jpg", "Sana 272.jpg", "Sana 276.jpg", "Sana 278.jpg", "Sana 288.jpg", "Sana 29.jpg", "Sana 293.jpg", "Sana 294.jpg", "Sana 295.jpg", "Sana 307.jpg", "Sana 311.jpg", "Sana 313.jpg", "Sana 32.jpg", "Sana 323.jpg", "Sana 332.jpg", "Sana 339.jpg", "Sana 34.jpg", "Sana 340.jpg", "Sana 345.jpg", "Sana 346.jpg", "Sana 347.jpg", "Sana 349.jpg", "Sana 35.jpg", "Sana 350.jpg", "Sana 351.jpg", "Sana 36.jpg", "Sana 361.jpg", "Sana 369.jpg", "Sana 37.jpg", "Sana 374.jpg", "Sana 375.jpg", "Sana 377.jpg", "Sana 380.jpg", "Sana 383.jpg", "Sana 384.jpg", "Sana 386.jpg", "Sana 387.jpg", "Sana 388.jpg", "Sana 389.jpg", "Sana 39.jpg", "Sana 392.jpg", "Sana 393.jpg", "Sana 395.jpg", "Sana 399.jpg", "Sana 40.jpg", "Sana 400.jpg", "Sana 401.jpg", "Sana 408.jpg", "Sana 410.jpg", "Sana 411.jpg", "Sana 413.jpg", "Sana 414.jpg", "Sana 419.jpg", "Sana 421.jpg", "Sana 423.jpg", "Sana 425.jpg", "Sana 426.jpg", "Sana 427.jpg", "Sana 43.jpg", "Sana 432.jpg", "Sana 436.jpg", "Sana 438.jpg", "Sana 442.jpg", "Sana 443.jpg", "Sana 446.jpg", "Sana 453.jpg", "Sana 455.jpg", "Sana 457.jpg", "Sana 46.jpg", "Sana 462.jpg", "Sana 464.jpg", "Sana 466.jpg", "Sana 469.jpg", "Sana 473.jpg", "Sana 476.jpg", "Sana 477.jpg", "Sana 48.jpg", "Sana 481.jpg", "Sana 486.jpg", "Sana 488.jpg", "Sana 489.jpg", "Sana 492.jpg", "Sana 496.jpg", "Sana 498.jpg", "Sana 500.jpg", "Sana 507.jpg", "Sana 509.jpg", "Sana 51.jpg", "Sana 511.jpg", "Sana 512.jpg", "Sana 513.jpg", "Sana 514.jpg", "Sana 515.jpg", "Sana 519.jpg", "Sana 52.jpg", "Sana 525.jpg", "Sana 530.jpg", "Sana 531.jpg", "Sana 532.jpg", "Sana 533.jpg", "Sana 536.jpg", "Sana 537.jpg", "Sana 539.jpg", "Sana 548.jpg", "Sana 549.jpg", "Sana 55.jpg", "Sana 551.jpg", "Sana 553.jpg", "Sana 555.jpg", "Sana 556.jpg", "Sana 557.jpg", "Sana 560.jpg", "Sana 565.jpg", "Sana 569.jpg", "Sana 57.jpg", "Sana 572.jpg", "Sana 575.jpg", "Sana 576.jpg", "Sana 577.jpg", "Sana 580.jpg", "Sana 581.jpg", "Sana 582.jpg", "Sana 583.jpg", "Sana 585.jpg", "Sana 586.jpg", "Sana 588.jpg", "Sana 59.jpg", "Sana 591.jpg", "Sana 598.jpg", "Sana 6.jpg", "Sana 60.jpg", "Sana 601.jpg", "Sana 602.jpg", "Sana 608.jpg", "Sana 610.jpg", "Sana 614.jpg", "Sana 617.jpg", "Sana 619.jpg", "Sana 621.jpg", "Sana 622.jpg", "Sana 624.jpg", "Sana 626.jpg", "Sana 629.jpg", "Sana 63.jpg", "Sana 634.jpg", "Sana 637.jpg", "Sana 638.jpg", "Sana 64.jpg", "Sana 643.jpg", "Sana 647.jpg", "Sana 651.jpg", "Sana 653.jpg", "Sana 655.jpg", "Sana 656.jpg", "Sana 657.jpg", "Sana 663.jpg", "Sana 667.jpg", "Sana 668.jpg", "Sana 67.jpg", "Sana 671.jpg", "Sana 672.jpg", "Sana 674.jpg", "Sana 677.jpg", "Sana 682.jpg", "Sana 685.jpg", "Sana 69.jpg", "Sana 691.jpg", "Sana 694.jpg", "Sana 696.jpg", "Sana 7.jpg", "Sana 70.jpg", "Sana 700.jpg", "Sana 701.jpg", "Sana 705.jpg", "Sana 708.jpg", "Sana 715.jpg", "Sana 717.jpg", "Sana 720.jpg", "Sana 723.jpg", "Sana 725.jpg", "Sana 728.jpg", "Sana 73.jpg", "Sana 731.jpg", "Sana 737.jpg", "Sana 739.jpg", "Sana 74.jpg", "Sana 741.jpg", "Sana 743.jpg", "Sana 75.jpg", "Sana 750.jpg", "Sana 752.jpg", "Sana 755.jpg", "Sana 757.jpg", "Sana 762.jpg", "Sana 766.jpg", "Sana 771.jpg", "Sana 775.jpg", "Sana 776.jpg", "Sana 777.jpg", "Sana 779.jpg", "Sana 78.jpg", "Sana 786.jpg", "Sana 788.jpg", "Sana 790.jpg", "Sana 794.jpg", "Sana 795.jpg", "Sana 80.jpg", "Sana 803.jpg", "Sana 808.jpg", "Sana 809.jpg", "Sana 81.jpg", "Sana 813.jpg", "Sana 814.jpg", "Sana 816.jpg", "Sana 817.jpg", "Sana 818.jpg", "Sana 819.jpg", "Sana 820.jpg", "Sana 822.jpg", "Sana 824.jpg", "Sana 827.jpg", "Sana 830.jpg", "Sana 839.jpg", "Sana 84.jpg", "Sana 841.jpg", "Sana 842.jpg", "Sana 843.jpg", "Sana 844.jpg", "Sana 845.jpg", "Sana 846.jpg", "Sana 847.jpg", "Sana 853.jpg", "Sana 857.jpg", "Sana 86.jpg", "Sana 860.jpg", "Sana 861.jpg", "Sana 863.jpg", "Sana 87.jpg", "Sana 873.jpg", "Sana 875.jpg", "Sana 877.jpg", "Sana 886.jpg", "Sana 89.jpg", "Sana 890.jpg", "Sana 892.jpg", "Sana 900.jpg", "Sana 902.jpg", "Sana 903.jpg", "Sana 905.jpg", "Sana 906.jpg", "Sana 91.jpg", "Sana 912.jpg", "Sana 913.jpg", "Sana 917.jpg", "Sana 919.jpg", "Sana 921.jpg", "Sana 928.jpg", "Sana 931.jpg", "Sana 932.jpg", "Sana 934.jpg", "Sana 935.jpg", "Sana 937.jpg", "Sana 938.jpg", "Sana 94.jpg", "Sana 940.jpg", "Sana 943.jpg", "Sana 951.jpg", "Sana 953.jpg", "Sana 956.jpg", "Sana 958.jpg", "Sana 960.jpg", "Sana 966.jpg", "Sana 968.jpg", "Sana 972.jpg", "Sana 976.jpg", "Sana 979.jpg", "Sana 983.jpg", "Sana 99.jpg", "Sana 991.jpg", "Sana 993.jpg"]}, "validation": {"ojo_de_gallo": ["20251129_092009.jpg", "20251129_092333.jpg", "20251129_092735.jpg", "20251129_092925.jpg", "20251129_093509.jpg", "20251129_093848.jpg", "20251129_094057.jpg", "20251129_094345.jpg", "20251129_094716.jpg", "20251129_095022.jpg", "20251129_095346.jpg", "20251129_095547.jpg", "20251129_095626.jpg", "20251129_095703.jpg", "20251129_095949.jpg", "20251129_100037.jpg", "20251129_100540.jpg", "20251129_100648.jpg", "20251129_100822.jpg", "20251129_101002.jpg", "20251129_101038.jpg", "20251129_101044.jpg", "20251129_101243.jpg", "20251129_101733.jpg", "20251129_102732.jpg", "20251129_102831.jpg", "20251129_103110.jpg", "20251129_103945.jpg", "20251129_104403.jpg", "20251129_104738.jpg", "20251129_104907.jpg", "20251129_104918.jpg", "20251129_105430.jpg", "20251129_105445.jpg", "20251129_114552.jpg", "20251129_114708.jpg", "20251129_114714.jpg", "20251129_114939.jpg", "20251129_114946.jpg", "20251129_115257.jpg", "20251129_115342.jpg", "20251129_115513.jpg", "20251129_115618.jpg", "20251129_115807.jpg", "20251129_120332.jpg", "20251129_120401.jpg", "20251129_120633.jpg", "20251129_120920.jpg", "20251129_130229.jpg", "20251129_130232.jpg", "20251129_130322.jpg", "20251129_130533.jpg", "20251129_130657.jpg", "20251129_131129.jpg", "20251129_131356.jpg", "20251129_131402.jpg", "20251129_131638.jpg", "20251129_132216.jpg", "20251129_132305.jpg", "20251129_132632.jpg", "20251129_132910.jpg", "20251129_133030.jpg", "20251129_133527.jpg", "20251129_133840.jpg", "20251129_133933.jpg", "20251129_133943.jpg", "20251129_134656.jpg", "20251129_134708.jpg", "20251129_134753.jpg", "20251129_135137.jpg", "20251129_135531.jpg", "20251129_135606.jpg", "20251129_135624.jpg", "20251129_135927.jpg", "20251129_140749.jpg"], "sana": ["Sana 135.jpg", "Sana 144.jpg", "Sana 155.jpg", "Sana 157.jpg", "Sana 161.jpg", "Sana 19.jpg", "Sana 197.jpg", "Sana 229.jpg", "Sana 239.jpg", "Sana 252.jpg", "Sana 257.jpg", "Sana 297.jpg", "Sana 300.jpg", "Sana 314.jpg", "Sana 316.jpg", "Sana 342.jpg", "Sana 352.jpg", "Sana 355.jpg", "Sana 356.jpg", "Sana 359.jpg", "Sana 379.jpg", "Sana 398.jpg", "Sana 403.jpg", "Sana 428.jpg", "Sana 435.jpg", "Sana 441.jpg", "Sana 467.jpg", "Sana 471.jpg", "Sana 472.jpg", "Sana 50.jpg", "Sana 524.jpg", "Sana 526.jpg", "Sana 528.jpg", "Sana 558.jpg", "Sana 567.jpg", "Sana 574.jpg", "Sana 592.jpg", "Sana 596.jpg", "Sana 611.jpg", "Sana 62.jpg", "Sana 632.jpg", "Sana 633.jpg", "Sana 652.jpg", "Sana 675.jpg", "Sana 684.jpg", "Sana 687.jpg", "Sana 693.jpg", "Sana 697.jpg", "Sana 703.jpg", "Sana 718.jpg", "Sana 72.jpg", "Sana 721.jpg", "Sana 749.jpg", "Sana 765.jpg", "Sana 781.jpg", "Sana 805.jpg", "Sana 826.jpg", "Sana 828.jpg", "Sana 856.jpg", "Sana 866.jpg", "Sana 876.jpg", "Sana 879.jpg", "Sana 882.jpg", "Sana 897.jpg", "Sana 910.jpg", "Sana 936.jpg", "Sana 939.jpg", "Sana 950.jpg", "Sana 957.jpg", "Sana 962.jpg", "Sana 974.jpg", "Sana 980.jpg", "Sana 987.jpg", "Sana 994.jpg"]}, "test": {"ojo_de_gallo": ["20251129_091427.jpg", "20251129_092633.jpg", "20251129_092724.jpg", "20251129_093405.jpg", "20251129_094358.jpg", "20251129_094634.jpg", "20251129_094823.jpg", "20251129_094932.jpg", "20251129_095649.jpg", "20251129_095825.jpg", "20251129_095836.jpg", "20251129_100024.jpg", "20251129_100529.jpg", "20251129_101314.jpg", "20251129_101826.jpg", "20251129_101959.jpg", "20251129_102055.jpg", "20251129_102103.jpg", "20251129_102113.jpg", "20251129_102445.jpg", "20251129_102558.jpg", "20251129_102920.jpg", "20251129_103018.jpg", "20251129_103228.jpg", "20251129_103610.jpg", "20251129_103921.jpg", "20251129_104056.jpg", "20251129_104456.jpg", "20251129_104554.jpg", "20251129_104647.jpg", "20251129_104818.jpg", "20251129_104945.jpg", "20251129_105017.jpg", "20251129_105036.jpg", "20251129_115401.jpg", "20251129_115438.jpg", "20251129_115623.jpg", "20251129_115857.jpg", "20251129_120211.jpg", "20251129_120253.jpg", "20251129_120538.jpg", "20251129_120659.jpg", "20251129_120717.jpg", "20251129_120953.jpg", "20251129_121103.jpg", "20251129_130241.jpg", "20251129_130419.jpg", "20251129_131111.jpg", "20251129_131229.jpg", "20251129_131241.jpg", "20251129_131303.jpg", "20251129_131351.jpg", "20251129_131423.jpg", "20251129_131620.jpg", "20251129_131753.jpg", "20251129_131923.jpg", "20251129_132248.jpg", "20251129_132523.jpg", "20251129_132727.jpg", "20251129_132740.jpg", "20251129_132859.jpg", "20251129_133824.jpg", "20251129_134344.jpg", "20251129_134917.jpg", "20251129_135125.jpg", "20251129_135900.jpg", "20251129_140025.jpg", "20251129_140043.jpg", "20251129_140236.jpg", "20251129_140508.jpg", "20251129_140516.jpg", "20251129_140753.jpg", "20251129_140759.jpg", "20251129_140802.jpg", "20251129_140813.jpg"], "sana": ["Sana 104.jpg", "Sana 119.jpg", "Sana 12.jpg", "Sana 122.jpg", "Sana 123.jpg", "Sana 133.jpg", "Sana 177.jpg", "Sana 187.jpg", "Sana 191.jpg", "Sana 193.jpg", "Sana 198.jpg", "Sana 204.jpg", "Sana 215.jpg", "Sana 23.jpg", "Sana 248.jpg", "Sana 250.jpg", "Sana 275.jpg", "Sana 279.jpg", "Sana 308.jpg", "Sana 309.jpg", "Sana 31.jpg", "Sana 312.jpg", "Sana 320.jpg", "Sana 341.jpg", "Sana 354.jpg", "Sana 360.jpg", "Sana 365.jpg", "Sana 382.jpg", "Sana 417.jpg", "Sana 420.jpg", "Sana 422.jpg", "Sana 431.jpg", "Sana 433.jpg", "Sana 45.jpg", "Sana 452.jpg", "Sana 490.jpg", "Sana 494.jpg", "Sana 516.jpg", "Sana 527.jpg", "Sana 571.jpg", "Sana 594.jpg", "Sana 600.jpg", "Sana 603.jpg", "Sana 615.jpg", "Sana 627.jpg", "Sana 635.jpg", "Sana 636.jpg", "Sana 649.jpg", "Sana 650.jpg", "Sana 660.jpg", "Sana 670.jpg", "Sana 68.jpg", "Sana 689.jpg", "Sana 711.jpg", "Sana 730.jpg", "Sana 732.jpg", "Sana 735.jpg", "Sana 745.jpg", "Sana 761.jpg", "Sana 77.jpg", "Sana 774.jpg", "Sana 79.jpg", "Sana 792.jpg", "Sana 832.jpg", "Sana 837.jpg", "Sana 838.jpg", "Sana 850.jpg", "Sana 869.jpg", "Sana 894.jpg", "Sana 914.jpg", "Sana 933.jpg", "Sana 959.jpg", "Sana 965.jpg", "Sana 97.jpg", "Sana 977.jpg", "Sana 990.jpg"]}}''')
ROYA = listar(RUTAS["roya"]); idx_r = np.random.RandomState(SEMILLA).permutation(len(ROYA))
a, b = int(round(.7 * len(ROYA))), int(round(.7 * len(ROYA))) + int(round(.15 * len(ROYA)))
div_r = {"train": idx_r[:a], "validation": idx_r[a:b], "test": idx_r[b:]}      # misma partición que la Fase 2c
MEN = {}
for p in ["train", "validation", "test"]:
    MEN[("ojo_de_gallo", p)] = leer_varias([os.path.join(RUTAS["ojo_de_gallo"], f) for f in PARTICION[p]["ojo_de_gallo"]], f"ojo {p}")[0]
    MEN[("sana", p)] = leer_varias([os.path.join(RUTAS["sana"], f) for f in PARTICION[p]["sana"]], f"sana {p}")[0]
    MEN[("roya", p)] = leer_varias([ROYA[i] for i in div_r[p]], f"roya {p}")[0]
PG = {}
for c, clase in [("ojo", "ojo_de_gallo"), ("healthy", "sana"), ("roya", "roya")]:
    X, rutas = leer_varias(listar(os.path.join(RUTAS["pg26038"], c)), f"PG {c}")
    hojas = [os.path.basename(r).split("__")[0] for r in rutas]
    PG[clase] = (X, hojas, [os.path.basename(r) for r in rutas])
    print(f"  {clase}: {len(X)} imágenes de {len(set(hojas))} hojas")

# Etiqueta del modelo: 1 = NO es ojo de gallo (sana o roya), 0 = ojo de gallo.
# Así la salida conserva el sentido del modelo v1 (P(no ojo)) y la aplicación calcula P(ojo) = 1 − salida.
def etiqueta(clase): return 0 if clase == "ojo_de_gallo" else 1
def mendeley(p):
    X = np.concatenate([MEN[(c, p)] for c in ["ojo_de_gallo", "sana", "roya"]])
    y = np.concatenate([np.full(len(MEN[(c, p)]), etiqueta(c)) for c in ["ojo_de_gallo", "sana", "roya"]]).astype("float32")
    return X, y

ojo train: 350 de 350 en 94 s
sana train: 349 de 349 en 93 s
roya train: 350 de 350 en 108 s
ojo validation: 75 de 75 en 20 s
sana validation: 74 de 74 en 20 s
roya validation: 75 de 75 en 28 s
ojo test: 75 de 75 en 20 s
sana test: 76 de 76 en 21 s
dañada: Roya 752.jpg broken data stream when reading image file
roya test: 74 de 75 en 22 s
PG ojo: 40 de 40 en 6 s
  ojo_de_gallo: 40 imágenes de 20 hojas
PG healthy: 20 de 20 en 3 s
  sana: 20 imágenes de 10 hojas
PG roya: 40 de 40 en 5 s
  roya: 40 imágenes de 20 hojas


## 2. Pliegues por hoja de PG 26038 (5 rondas, estratificados por clase)

In [3]:
rng = np.random.RandomState(SEMILLA); PLIEGUE = {}
for clase, (X, hojas, _) in PG.items():
    u = sorted(set(hojas)); rng.shuffle(u)
    for i, h in enumerate(u): PLIEGUE[(clase, h)] = i % K
pd.DataFrame([{"clase": c, "hoja": h, "pliegue": k} for (c, h), k in PLIEGUE.items()]).to_csv(
    os.path.join(SAL, "tablas", "pliegues_pg26038.csv"), index=False)
def pg_subconjunto(pliegues):
    Xs, ys, info = [], [], []
    for clase, (X, hojas, nombres) in PG.items():
        m = np.array([PLIEGUE[(clase, h)] in pliegues for h in hojas])
        if m.any():
            Xs.append(X[m]); ys.append(np.full(m.sum(), etiqueta(clase)))
            info += [(clase, h, n) for h, n, mm in zip(hojas, nombres, m) if mm]
    return np.concatenate(Xs), np.concatenate(ys).astype("float32"), info

## 3. Modelo, entrenamiento y promedio de cuatro vistas

In [4]:
def construir():
    fijar_semillas()
    e = keras.Input((TAM, TAM, 3), name="imagen")
    aum = keras.Sequential([keras.layers.RandomFlip("horizontal_and_vertical"), keras.layers.RandomRotation(0.25),
                            keras.layers.RandomZoom(0.15), keras.layers.RandomContrast(0.2),
                            keras.layers.RandomBrightness(0.2, value_range=(0, 255))], name="data_augmentation")
    base = keras.applications.MobileNetV3Small(input_shape=(TAM, TAM, 3), include_top=False,
                                               weights=None if MOCK else "imagenet", include_preprocessing=True)
    base.trainable = False
    z = base(aum(e), training=False)
    z = keras.layers.GlobalAveragePooling2D(name="global_average_pooling")(z)
    z = keras.layers.Dropout(0.2, name="dropout")(z)
    z = keras.layers.Dense(1, activation="sigmoid", name="clasificacion")(z)
    m = keras.Model(e, z); m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy")
    return m

def entrenar(X, y, Xv, yv):
    m = construir()
    pesos = {0: len(y) / (2 * (y == 0).sum()), 1: len(y) / (2 * (y == 1).sum())}
    ruta = os.path.join(TMP, "mejor.weights.h5")
    m.fit(ds_desde(X, y, True), validation_data=ds_desde(Xv, yv, False), epochs=EPOCAS, class_weight=pesos, verbose=0,
          callbacks=[keras.callbacks.ModelCheckpoint(ruta, monitor="val_loss", save_best_only=True, save_weights_only=True)])
    m.load_weights(ruta)
    ent = keras.Input((TAM, TAM, 3), name="imagen"); x = ent
    for capa in m.layers[1:]:
        if capa.name != "data_augmentation": x = capa(x, training=False)
    return keras.Model(ent, x)

def vistas(X):
    return [X, X[:, :, ::-1], X[:, ::-1, :], X[:, ::-1, ::-1]]   # original, espejo horizontal, vertical, 180°
def p_ojo(m, X, tta):
    if not tta: return 1 - m.predict(ds_desde(X, np.zeros(len(X)), False), verbose=0).ravel()
    return np.mean([1 - m.predict(ds_desde(np.ascontiguousarray(v), np.zeros(len(X)), False), verbose=0).ravel() for v in vistas(X)], axis=0)

class V1:
    """Diagnóstico actual de la aplicación (TFLite): salida = P(sana)."""
    def __init__(self, ruta):
        self.it = tf.lite.Interpreter(model_path=ruta); self.it.allocate_tensors()
        self.i = self.it.get_input_details()[0]; self.o = self.it.get_output_details()[0]
    def predict(self, X):
        s = []
        for k in range(len(X)):
            self.it.set_tensor(self.i["index"], X[k:k + 1].astype("float32")); self.it.invoke()
            s.append(float(self.it.get_tensor(self.o["index"]).ravel()[0]))
        return np.array(s)
v1 = V1(RUTAS["diag_v1"]) if os.path.isfile(RUTAS["diag_v1"]) else None
def p_ojo_v1(X, tta):
    if not tta: return 1 - v1.predict(X)
    return np.mean([1 - v1.predict(np.ascontiguousarray(v)) for v in vistas(X)], axis=0)

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


## 4. Validación cruzada por hoja (las hojas evaluadas nunca se usaron para entrenar)

In [5]:
Xm_tr, ym_tr = mendeley("train"); Xm_va, ym_va = mendeley("validation")
filas = []
for k in range(K):
    Xp_tr, yp_tr, _ = pg_subconjunto([j for j in range(K) if j != k])
    Xp_te, yp_te, info = pg_subconjunto([k])
    t0 = time.time()
    m = entrenar(np.concatenate([Xm_tr, Xp_tr]), np.concatenate([ym_tr, yp_tr]), Xm_va, ym_va)
    for tta in [False, True]:
        pn = p_ojo(m, Xp_te, tta)
        pv = p_ojo_v1(Xp_te, tta) if v1 else np.full(len(pn), np.nan)
        for (clase, hoja, nombre), a_, b_ in zip(info, pn, pv):
            filas.append({"pliegue": k, "tta": tta, "clase": clase, "hoja": hoja, "imagen": nombre, "p_ojo_v3": a_, "p_ojo_v1": b_})
    print(f"Ronda {k + 1}/{K}: {len(info)} imágenes de prueba, {time.time() - t0:.0f} s")
CV = pd.DataFrame(filas); CV.to_csv(os.path.join(SAL, "tablas", "cv_pg26038_por_imagen.csv"), index=False)

4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Ronda 1/5: 20 imágenes de prueba, 56 s
Ronda 2/5: 20 imágenes de prueba, 37 s
Ronda 3/5: 20 imágenes de prueba, 37 s
Ronda 4/5: 20 imágenes de prueba, 37 s
Ronda 5/5: 20 imágenes de prueba, 37 s


## 5. Resultados: errores del modelo y de la regla de la aplicación
Falso negativo = hoja con ojo de gallo que no se marca; falso positivo = hoja sana o con roya marcada como ojo de gallo.
Regla de la aplicación: P(ojo) ≥ 0,70 compatible; ≤ 0,30 sin síntomas; en medio, no concluyente.

In [6]:
def resumen(df, col, nombre):
    es_ojo = (df.clase == "ojo_de_gallo").values; p = df[col].values
    vp = int((p[es_ojo] >= .5).sum()); vn = int((p[~es_ojo] < .5).sum())
    roya = (df.clase == "roya").values
    regla = np.where(p >= .7, "compatible", np.where(p <= .3, "sin_sintomas", "no_concluyente"))
    concl = regla != "no_concluyente"
    err_concl = int(((regla == "compatible") & ~es_ojo).sum() + ((regla == "sin_sintomas") & es_ojo).sum())
    return {"modelo": nombre, "n_ojo": int(es_ojo.sum()), "n_no_ojo": int((~es_ojo).sum()),
            "sensibilidad": vp / es_ojo.sum(), "sens_IC95": wilson(vp, int(es_ojo.sum())),
            "especificidad": vn / (~es_ojo).sum(), "esp_IC95": wilson(vn, int((~es_ojo).sum())),
            "falsos_negativos": int(es_ojo.sum() - vp), "falsos_positivos": int((~es_ojo).sum() - vn),
            "roya_como_ojo": float((p[roya] >= .5).mean()) if roya.any() else float("nan"),
            "auc": roc_auc_score(es_ojo, p),
            "regla_no_concluyente": float((~concl).mean()),
            "regla_errores_entre_concluyentes": err_concl, "regla_concluyentes": int(concl.sum())}

def por_hoja(df, col):
    g = df.groupby(["clase", "hoja"])[col].mean().reset_index(); g["hoja_id"] = g.hoja
    return g

RES = []
for tta in [False, True]:
    d = CV[CV.tta == tta]
    for col, nm in [("p_ojo_v3", "v3 ojo-vs-resto + PG (CV)"), ("p_ojo_v1", "v1 actual")]:
        if d[col].isna().all(): continue
        r = resumen(d, col, nm); r["tta"] = tta; r["nivel"] = "imagen"; RES.append(r)
        r = resumen(por_hoja(d, col), col, nm); r["tta"] = tta; r["nivel"] = "hoja"; RES.append(r)
df_res = pd.DataFrame(RES); df_res.to_csv(os.path.join(SAL, "tablas", "resumen_cv_pg26038.csv"), index=False)
cols = ["nivel", "tta", "modelo", "sensibilidad", "especificidad", "falsos_negativos", "falsos_positivos", "roya_como_ojo",
        "auc", "regla_no_concluyente", "regla_errores_entre_concluyentes", "regla_concluyentes"]
print(df_res[cols].round(3).to_string(index=False))

 nivel   tta                    modelo  sensibilidad  especificidad  falsos_negativos  falsos_positivos  roya_como_ojo   auc  regla_no_concluyente  regla_errores_entre_concluyentes  regla_concluyentes
imagen False v3 ojo-vs-resto + PG (CV)         0.750          0.450                10                33          0.625 0.749                  0.62                                 9                  38
  hoja False v3 ojo-vs-resto + PG (CV)         0.750          0.533                 5                14          0.550 0.753                  0.66                                 3                  17
imagen False                 v1 actual         0.650          0.850                14                 9          0.200 0.816                  0.38                                 9                  62
  hoja False                 v1 actual         0.650          0.867                 7                 4          0.200 0.852                  0.46                                 3                

## 6. Modelo final (Perú + todas las hojas de PG 26038) y prueba de Mendeley
La prueba de Mendeley (225 imágenes, tres clases) no se usó para entrenar ni para elegir el modelo.

In [7]:
Xp_all, yp_all, _ = pg_subconjunto(list(range(K)))
final = entrenar(np.concatenate([Xm_tr, Xp_all]), np.concatenate([ym_tr, yp_all]), Xm_va, ym_va)
Xm_te = np.concatenate([MEN[(c, "test")] for c in ["ojo_de_gallo", "sana", "roya"]])
clase_te = sum([[c] * len(MEN[(c, "test")]) for c in ["ojo_de_gallo", "sana", "roya"]], [])
MT = []
for tta in [False, True]:
    for nm, fn in [("v3 final", lambda X, t: p_ojo(final, X, t)), ("v1 actual", p_ojo_v1 if v1 else None)]:
        if fn is None: continue
        d = pd.DataFrame({"clase": clase_te, "hoja": range(len(clase_te)), "p": fn(Xm_te, tta)})
        r = resumen(d, "p", nm); r["tta"] = tta; MT.append(r)
df_mt = pd.DataFrame(MT); df_mt.to_csv(os.path.join(SAL, "tablas", "mendeley_prueba_v1_v3.csv"), index=False)
print(df_mt[cols[1:]].round(3).to_string(index=False))

  tta    modelo  sensibilidad  especificidad  falsos_negativos  falsos_positivos  roya_como_ojo   auc  regla_no_concluyente  regla_errores_entre_concluyentes  regla_concluyentes
False  v3 final           1.0          0.993                 0                 1          0.014 0.999                 0.022                                 1                 220
False v1 actual           1.0          0.773                 0                34          0.446 0.974                 0.058                                26                 212
 True  v3 final           1.0          0.993                 0                 1          0.014 1.000                 0.027                                 0                 219
 True v1 actual           1.0          0.793                 0                31          0.419 0.977                 0.062                                25                 211


## 7. Conversión a TensorFlow Lite (float16) para la aplicación

In [8]:
carpeta = os.path.join(TMP, "sm_final")
try: final.export(carpeta, verbose=False)
except TypeError: final.export(carpeta)
c = tf.lite.TFLiteConverter.from_saved_model(carpeta)
c.optimizations = [tf.lite.Optimize.DEFAULT]; c.target_spec.supported_types = [tf.float16]
B = c.convert()
it = tf.lite.Interpreter(model_content=B); it.allocate_tensors(); i_ = it.get_input_details()[0]; o_ = it.get_output_details()[0]
s = []
for k in range(len(Xm_te)):
    it.set_tensor(i_["index"], Xm_te[k:k + 1].astype("float32")); it.invoke(); s.append(float(it.get_tensor(o_["index"]).ravel()[0]))
ref = final.predict(ds_desde(Xm_te, np.zeros(len(Xm_te)), False), verbose=0).ravel()
conv = {"MB": len(B) / 1e6, "acuerdo_decision": float(((np.array(s) >= .5) == (ref >= .5)).mean()), "dif_max": float(np.abs(np.array(s) - ref).max()),
        "salida": "P(no es ojo de gallo); la app usa P(ojo) = 1 - salida"}
print(conv); guardar_json(conv, "conversion_tflite.json")
open(os.path.join(SAL, "android", "modelo_diagnostico.tflite"), "wb").write(B)
guardar_json({"versiones": VERSIONES, "epocas": EPOCAS, "pliegues": K}, "resumen_fase2d.json")
print("Modelo para la app en", os.path.join(SAL, "android"))

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


{'MB': 1.944604, 'acuerdo_decision': 1.0, 'dif_max': 0.01263493299484253, 'salida': 'P(no es ojo de gallo); la app usa P(ojo) = 1 - salida'}
Modelo para la app en /content/drive/MyDrive/tesis_ojo_de_gallo/fase2d_mitigacion/android
